# 🚀 02: Model Training, Cross-Validation & Benchmarking Pipeline
### YouTube Comment Sentiment Classifier
---
**Objective:** Benchmark multiple machine learning models (Naive Bayes, Logistic Regression, LinearSVC with Probability Calibration, SGD, Voting Ensemble) with TF-IDF sublinear n-gram feature representations, evaluate with detailed metrics, and export the production pipeline.

In [ ]:
# 1. Imports & Configuration
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.trainer import ModelTrainer
from src.models.predictor import SentimentPredictor
from src.utils.config_manager import load_config

config = load_config('../configs/config.yaml')

In [ ]:
# 2. Execute Full Training and Benchmarking Pipeline
trainer = ModelTrainer(config)
best_pipeline, benchmark_results = trainer.train_and_benchmark()
print(f'Best Selected Architecture: {benchmark_results.get("best_model")}')
print(f'Best Macro F1: {benchmark_results.get("best_score"):.4f}')

In [ ]:
# 3. Model Benchmark Comparison Table
models_dict = benchmark_results['models']
rows = []
for name, m in models_dict.items():
    rows.append({
        'Architecture': name,
        'Accuracy': m['accuracy'],
        'Macro F1': m['f1_macro'],
        'Weighted F1': m['f1_weighted'],
        'Precision': m['precision_macro'],
        'Recall': m['recall_macro'],
        'Avg Latency (ms)': m.get('latency', {}).get('avg_latency_ms', 0.0)
    })
df_bench = pd.DataFrame(rows).sort_values('Macro F1', ascending=False)
print(df_bench.to_string(index=False))

In [ ]:
# 4. Confusion Matrix for Best Model
best_name = benchmark_results['best_model']
best_m = models_dict[best_name]
cm = best_m['confusion_matrix']
target_names = best_m['target_names']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title(f'Confusion Matrix: {best_name}', fontsize=14, weight='bold')
plt.xlabel('Predicted Sentiment')
plt.ylabel('True Sentiment')
plt.show()

In [ ]:
# 5. Real-Time Inference with Exported Production Pipeline
predictor = SentimentPredictor('../models/sentiment_pipeline.joblib', config=config)
test_comments = [
    'This video is extraordinary! Masterpiece quality.',
    'Can you share the dataset link?',
    'Worst video ever, completely misleading information.'
]
for comment in test_comments:
    res = predictor.predict(comment)
    print(f'Comment: \"{res["raw_text"]}\"')
    print(f'-> Prediction: {res["label"]} (Confidence: {res["confidence"]:.2%}, Latency: {res["latency_ms"]} ms)')
    print('-' * 50)